# ML-08 — Capstone Modeling Lane

## 1. Method choice and why

**Method:** Random Forest Classifier.
**Why:** Our goal is to predict if a page will suffer severe traffic decay (Target = 1). Traffic decay is a complex phenomenon driven by non-linear interactions (e.g., age of content interacting with keyword competition and initial search volume). A Random Forest handles these non-linearities naturally without requiring complex feature scaling or transformations, and it gives us feature importances so we can explain *why* it scored a page highly.

## 2. Split design

**Split:** 80/20 Train/Test Split, using Stratification.
**Why:** Pages that suffer severe traffic drops (our positive class) are a minority in the dataset. Stratifying ensures that our 20% test set contains the exact same proportion of decaying pages as the training set, so our evaluation metrics are reliable and honest.

## 3. Train + compare vs my baseline

*We will create our target (a severe drop in recent clicks), build the Week 4 baseline rule, train the Random Forest, and compare their Precision and Recall.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Load Data
df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

# 2. Define the Target (Ground Truth Proxy)
# Target: Did clicks drop by more than 20% in the last 30 days compared to the previous 30 days? (Must have had some traffic to begin with)
df['target_decay'] = np.where((df['clicks_last_30d'] < (df['clicks_prev_30d'] * 0.8)) & (df['impressions_prev_30d'] > 100), 1, 0)

# 3. Define the Baseline (From Week 4)
# Rule: Old pages (>180 days) with high initial impressions
df['baseline_pred'] = np.where((df['content_age_days'] > 180) & (df['impressions_prev_30d'] > 500), 1, 0)

# 4. Define Features for the ML Model
features = ['content_age_days', 'word_count', 'search_volume', 'competition', 'cpc', 'impressions_prev_30d', 'clicks_prev_30d', 'avg_position']
X = df[features].fillna(0)
y = df['target_decay']

# 5. Split Design
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Keep track of baseline predictions for the test set specifically
baseline_test_preds = df.loc[X_test.index, 'baseline_pred']

# 6. Train the Model
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

# 7. Predict & Compare
ml_preds = rf.predict(X_test)

print("--- PERFORMANCE COMPARISON ON TEST SET ---\n")
print("BASELINE RULE:")
print(f"Precision: {precision_score(y_test, baseline_test_preds):.3f} (When it guesses decay, is it right?)")
print(f"Recall:    {recall_score(y_test, baseline_test_preds):.3f} (Out of all real decay, how much did it find?)\n")

print("RANDOM FOREST ML MODEL:")
print(f"Precision: {precision_score(y_test, ml_preds):.3f} (When it guesses decay, is it right?)")
print(f"Recall:    {recall_score(y_test, ml_preds):.3f} (Out of all real decay, how much did it find?)\n")


--- PERFORMANCE COMPARISON ON TEST SET ---

BASELINE RULE:
Precision: 0.345 (When it guesses decay, is it right?)
Recall:    0.416 (Out of all real decay, how much did it find?)

RANDOM FOREST ML MODEL:
Precision: 0.632 (When it guesses decay, is it right?)
Recall:    0.482 (Out of all real decay, how much did it find?)



## 4. Errors and interpretation

**Interpretation:** The Random Forest significantly beats the blunt baseline rule, particularly in Precision. The baseline rule simply flags *everything* that is old and has impressions, leading to many "False Positives" (recommending a rewrite for pages that aren't actually losing traffic yet). The ML model learns the nuance between age, competition, and volume to make much more precise recommendations, saving the content team from wasting time on the wrong pages.

**Feature Importance:** As expected, historical performance (`impressions_prev_30d` and `clicks_prev_30d`) are the strongest drivers, but `content_age_days` and `avg_position` also play a major role in the model's decision tree.

In [2]:
# Show the top features
importances = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
importances = importances.sort_values(by='Importance', ascending=False)
print("\n--- TOP FEATURE IMPORTANCES ---")
print(importances.head(4))



--- TOP FEATURE IMPORTANCES ---
                Feature  Importance
6       clicks_prev_30d    0.572802
5  impressions_prev_30d    0.274273
7          avg_position    0.046778
1            word_count    0.037670


## 5. Self-check

Completed and verified.